In [1]:
import os
import sys
import napari

sys.path.append("..")  # Adds higher directory to python modules path.
from src import RASPRoutines
import polars as pl
import numpy as np
from src import IOFunctions
from src import AnalysisFunctions
from src import Image_Analysis_Functions
from src import HelperFunctions

RASP = RASPRoutines.RASP_Routines()
IO = IOFunctions.IO_Functions()
A_F = AnalysisFunctions.Analysis_Functions()
IA_F = Image_Analysis_Functions.ImageAnalysis_Functions()
H_F = HelperFunctions.Helper_Functions()

import warnings
from copy import copy

warnings.filterwarnings("ignore")

import zarr

from ome_zarr.io import parse_url
from ome_zarr.writer import write_image

In [2]:
path = "test_image_for_IDR_josephsbeckwith.zarr"

In [3]:
protein_image = IO.read_tiff("exemplar_protein_image.tiff")
cell_image = IO.read_tiff("exemplar_cell_image.tiff")
cell_mask_image = IO.read_tiff("exemplar_cell_mask_image.tiff")
subset = pl.read_csv("examplar_oligomer_locations.csv")

In [12]:
np.stack([protein_image, cell_image], axis=-1).shape

(25, 1200, 1200, 2)

In [15]:
import numpy as np
import os

from skimage.data import binary_blobs
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image

maxval = 4095

if os.path.isdir(path):
    import shutil

    shutil.rmtree(path)  # Use rmtree as zarr may create multiple files/dirs

os.mkdir(path)

axes = [
    {"name": "c", "type": "channel"},
    {"name": "z", "type": "space"},
    {"name": "x", "type": "space"},
    {"name": "y", "type": "space"},
]

size_xy = cell_image.shape[1]
size_z = cell_image.shape[0]

# write the image data
store = parse_url(path, mode="w").store
root = zarr.group(store=store)
write_image(
    image=np.stack([protein_image, cell_image]),
    group=root,
    storage_options=dict(chunks=(1, 1, size_xy, size_xy)),
    axes=axes,
    channel_names=["protein", "cell"],
)  # Add channel names

min_protein, max_protein = np.percentile(protein_image, (0.1, 99.9)).astype(int)
min_cell, max_cell = np.percentile(cell_image, (0.1, 99.9)).astype(int)

root.attrs["omero"] = {
    "channels": [
        {
            "color": "00FF00",
            "window": {"start": min_protein, "end": max_protein},
            "label": "protein",
            "active": True,
        },
        {
            "color": "FF0000",
            "window": {"start": min_cell, "end": max_cell},
            "label": "cell",
            "active": True,
        },
    ]
}
# add labels...
cell_mask = cell_mask_image.astype("int8")
protein_mask = np.zeros_like(cell_mask)
x = subset["x"].to_numpy().astype("int")
y = subset["y"].to_numpy().astype("int")
z = subset["z"].to_numpy().astype("int") - 1
for xval in np.arange(-2, 3).astype("int"):
    for yval in np.arange(-2, 3).astype("int"):
        protein_mask[z, x + xval, y + yval] = 2

# Write labels to separate groups
labels_grp = root.create_group("labels")

# 1. Cell Mask (values 1)
labels_grp.attrs["labels"] = ["cell_mask", "protein_mask"]  # Both labels listed

# Cell mask group
cell_label_grp = labels_grp.create_group("cell_mask")
cell_label_grp.attrs["image-label"] = {"version": "0.4", "source": {"image": "../../"}}
cell_label_grp.attrs["colors"] = [
    {"label-value": 1, "rgba": [255, 0, 0, 158]}  # Semi-transparent red
]
write_image(cell_mask, cell_label_grp, axes="zyx")

# 2. Protein Mask (values 2)
protein_label_grp = labels_grp.create_group("protein_mask")
protein_label_grp.attrs["image-label"] = {
    "version": "0.4",
    "source": {"image": "../../"},
}
protein_label_grp.attrs["colors"] = [
    {"label-value": 1, "rgba": [255, 255, 255, 255]}  # Solid white
]
write_image(protein_mask, protein_label_grp, axes="zyx")

# Force sync writes
store.close()

In [16]:
viewer = napari.Viewer()
viewer.open(path, plugin="napari-ome-zarr")

[<Image layer 'protein' at 0x7f60dedc9c40>,
 <Image layer 'cell' at 0x7f60de9b0ec0>,
 <Labels layer '/labels/cell_mask' at 0x7f60ddd60aa0>,
 <Labels layer '/labels/protein_mask' at 0x7f60dc269550>]

Traceback (most recent call last):
  File "/home/jsb92/pyRASP/lib/python3.12/site-packages/napari/_qt/qt_main_window.py", line 254, in set_status_and_tooltip
    self._qt_viewer.viewer.status = status_and_tooltip[0]
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jsb92/pyRASP/lib/python3.12/site-packages/napari/utils/events/evented_model.py", line 307, in __setattr__
    with ComparisonDelayer(self):
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jsb92/pyRASP/lib/python3.12/site-packages/napari/utils/events/evented_model.py", line 520, in __exit__
    self._target._check_if_values_changed_and_emit_if_needed()
  File "/home/jsb92/pyRASP/lib/python3.12/site-packages/napari/utils/events/evented_model.py", line 345, in _check_if_values_changed_and_emit_if_needed
    getattr(self.events, name)(value=new_value)
  File "/home/jsb92/pyRASP/lib/python3.12/site-packages/napari/utils/events/event.py", line 764, in __call__
    self._invoke_callback(cb, event if pass_event else None)
  File "/home